In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import average_precision_score, roc_auc_score

# 1. Load data and splits
curated = pd.read_csv("../data/curated_chembl_eligible.csv")
splits  = json.loads(Path("../splits/split_indices.json").read_text())

# Guard: the split must reference the eligible file, and no ambiguous rows
# may be present in the training/eval universe.
assert splits["curated_csv"] == "data/curated_chembl_eligible.csv", (
    "split_indices.json points at the wrong source file — re-run 01_data_curation.ipynb"
)
assert set(curated["activity_class"].unique()) <= {"active", "inactive"}, (
    f"ambiguous rows present: {curated['activity_class'].value_counts().to_dict()}"
)

# 2. Generate ECFP4 using the new API
# Locked: ECFP4 (radius=2), 2048 bits
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def generate_ecfp4(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = morgan_gen.GetFingerprint(mol)
    return list(fp)

curated["ecfp4"] = curated["canonical_smiles"].apply(generate_ecfp4)

# Check for any failures
missing_fps = curated["ecfp4"].isna().sum()
if missing_fps > 0:
    print(f"WARNING: {missing_fps} compounds failed to generate fingerprints.")
else:
    print("ECFP4 feature generation complete. No failures.")

# 3. Build matrices
def get_xy(indices):
    subset = curated.iloc[indices]
    X = np.array(subset["ecfp4"].tolist())
    y = (subset["activity_class"] == "active").astype(int).values
    return X, y

X_train, y_train = get_xy(splits["indices"]["train"])
X_val, y_val = get_xy(splits["indices"]["validation"])
X_test, y_test = get_xy(splits["indices"]["test"])

print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

ECFP4 feature generation complete. No failures.
Train shape: (128, 2048)
Validation shape: (30, 2048)
Test shape: (21, 2048)


In [2]:
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import average_precision_score

cv_data = json.loads(Path("../splits/cv_scaffold_groups.json").read_text())
cv_folds_global = cv_data["folds"]

train_global_indices = splits["indices"]["train"]
global_to_local = {g: l for l, g in enumerate(train_global_indices)}

cv_folds_local = {}
for fold_name, global_idxs in cv_folds_global.items():
    cv_folds_local[fold_name] = [
        global_to_local[g] for g in global_idxs if g in global_to_local
    ]

def evaluate_cv_per_fold(model_func, X, y, folds):
    """Evaluate per fold. Class weights computed from fold-train only (no leakage).
    Empty folds (0 actives) are recorded but excluded from the mean."""
    per_fold = []
    for fold_name, fold_local in folds.items():
        fold_set = set(fold_local)
        train_local = [i for i in range(len(X)) if i not in fold_set]

        y_fold_train = y[train_local]
        n_pos_train = int(y_fold_train.sum())
        n_neg_train = int(len(y_fold_train) - n_pos_train)

        y_fold_val = y[fold_local]
        n_pos_val = int(y_fold_val.sum())

        if n_pos_val == 0:
            per_fold.append({
                "fold": fold_name, "pr_auc": None,
                "n_val_actives": 0, "n_val_inactives": len(fold_local),
                "n_train_actives": n_pos_train, "n_train_inactives": n_neg_train,
                "note": "no positives in held-out fold",
            })
            continue

        model = model_func(n_pos_train, n_neg_train)
        model.fit(X[train_local], y_fold_train)
        preds = model.predict_proba(X[fold_local])[:, 1]
        ap = float(average_precision_score(y_fold_val, preds))

        per_fold.append({
            "fold": fold_name, "pr_auc": ap,
            "n_val_actives": n_pos_val, "n_val_inactives": len(fold_local) - n_pos_val,
            "n_train_actives": n_pos_train, "n_train_inactives": n_neg_train,
        })

    scored = [f["pr_auc"] for f in per_fold if f["pr_auc"] is not None]
    mean = float(np.mean(scored)) if scored else float("nan")
    std = float(np.std(scored)) if scored else float("nan")
    return mean, std, per_fold

# XGB: scale_pos_weight computed inside each fold
xgb_func = lambda n_pos, n_neg: xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1,
    scale_pos_weight=(n_neg / n_pos) if n_pos > 0 else 1.0,
    random_state=42, eval_metric="logloss",
)
xgb_mean, xgb_std, xgb_folds_detail = evaluate_cv_per_fold(xgb_func, X_train, y_train, cv_folds_local)

# RF: class_weight='balanced' is per-fit, no explicit weight needed
rf_func = lambda n_pos, n_neg: RandomForestClassifier(
    n_estimators=100, max_depth=5, class_weight="balanced", random_state=42,
)
rf_mean, rf_std, rf_folds_detail = evaluate_cv_per_fold(rf_func, X_train, y_train, cv_folds_local)

print("\n=== XGB CV per fold ===")
for f in xgb_folds_detail:
    print(f"  fold {f['fold']}: pr_auc={f['pr_auc']}, "
          f"train_act={f['n_train_actives']}, val_act={f['n_val_actives']}")
print(f"XGB mean (scored folds only): {xgb_mean:.3f} ± {xgb_std:.3f}")

print("\n=== RF CV per fold ===")
for f in rf_folds_detail:
    print(f"  fold {f['fold']}: pr_auc={f['pr_auc']}, "
          f"train_act={f['n_train_actives']}, val_act={f['n_val_actives']}")
print(f"RF mean (scored folds only): {rf_mean:.3f} ± {rf_std:.3f}")

# Model selection
if xgb_mean > rf_mean:
    print("\nSelected Model: XGBoost")
    selected_model = xgb_func(
        int(y_train.sum()), int(len(y_train) - y_train.sum())
    )
    selected_model_type = "XGBClassifier"
else:
    print("\nSelected Model: Random Forest")
    selected_model = rf_func(
        int(y_train.sum()), int(len(y_train) - y_train.sum())
    )
    selected_model_type = "RandomForestClassifier"

selected_model.fit(X_train, y_train)


=== XGB CV per fold ===
  fold 1: pr_auc=1.0, train_act=8, val_act=6
  fold 2: pr_auc=0.16, train_act=10, val_act=4
  fold 3: pr_auc=1.0, train_act=12, val_act=2
  fold 4: pr_auc=0.53125, train_act=12, val_act=2
XGB mean (scored folds only): 0.673 ± 0.353

=== RF CV per fold ===
  fold 1: pr_auc=1.0, train_act=8, val_act=6
  fold 2: pr_auc=1.0, train_act=10, val_act=4
  fold 3: pr_auc=1.0, train_act=12, val_act=2
  fold 4: pr_auc=0.75, train_act=12, val_act=2
RF mean (scored folds only): 0.938 ± 0.108

Selected Model: Random Forest


,n_estimators,100
,criterion,'gini'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [3]:
def evaluate_split(model, X, y, name):
    preds = model.predict_proba(X)[:, 1]
    pr_auc = average_precision_score(y, preds)
    roc_auc = roc_auc_score(y, preds)
    
    # Top-N recovery (N = number of actives)
    n_actives = sum(y)
    top_n_preds = np.argsort(preds)[-n_actives:]
    top_n_hits = sum(y[top_n_preds])
    top_n_recovery = top_n_hits / n_actives
    
    print(f"[{name}] N_actives: {n_actives}, N_inactives: {len(y)-n_actives}")
    print(f"  PR-AUC: {pr_auc:.3f}, ROC-AUC: {roc_auc:.3f}")
    print(f"  Top-{n_actives} Recovery: {top_n_recovery:.3f} ({top_n_hits}/{n_actives})")
    return pr_auc, roc_auc, top_n_recovery

print("--- Validation ---")
evaluate_split(selected_model, X_val, y_val, "Validation")
print("\n--- Test (Headline) ---")
evaluate_split(selected_model, X_test, y_test, "Test")

--- Validation ---
[Validation] N_actives: 10, N_inactives: 20
  PR-AUC: 0.981, ROC-AUC: 0.990
  Top-10 Recovery: 0.900 (9/10)

--- Test (Headline) ---
[Test] N_actives: 7, N_inactives: 14
  PR-AUC: 0.781, ROC-AUC: 0.760
  Top-7 Recovery: 0.714 (5/7)


(0.7806122448979591, 0.760204081632653, np.float64(0.7142857142857143))

In [4]:
import joblib
import json
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score

# ---------- 1. Save model and config ----------
Path("../models").mkdir(exist_ok=True)

model_path = (
    "../models/selected_model_rf.pkl"
    if selected_model_type == "RandomForestClassifier"
    else "../models/selected_model_xgb.pkl"
)
joblib.dump(selected_model, model_path)
print(f"Model saved to {model_path}")

model_config = {
    "schema_version": "1.2.0",
    "selected_model": selected_model_type,
    "xgb_cv_pr_auc": float(xgb_mean),
    "rf_cv_pr_auc": float(rf_mean),
    "xgb_cv_per_fold": xgb_folds_detail,
    "rf_cv_per_fold": rf_folds_detail,
    "cv_n_folds": len(cv_folds_local),
    "cv_folds_with_zero_actives": [
        f["fold"] for f in rf_folds_detail if f["pr_auc"] is None
    ],
    "cv_scored_folds_n": sum(1 for f in rf_folds_detail if f["pr_auc"] is not None),
    "selected_model_params": (
        {"n_estimators": 100, "max_depth": 5, "class_weight": "balanced",
         "random_state": 42}
        if selected_model_type == "RandomForestClassifier"
        else {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1,
              "random_state": 42,
              "scale_pos_weight": float((len(y_train) - y_train.sum()) / y_train.sum())}
    ),
    "training_universe": "data/curated_chembl_eligible.csv (active + inactive only)",
    "scaffold_definition": "bemis_murcko_canonical_no_chirality",
}
Path("../models/model_config.json").write_text(json.dumps(model_config, indent=2))
print("Model config saved to ../models/model_config.json")


# ---------- 2. Protocol §5.3 small-N flag derivation ----------
def small_n_flags(n_actives: int) -> dict:
    """Deterministic flags from test N_actives, per protocol.md §5.3."""
    return {
        "top_recovery_mode":  "top_N" if n_actives < 20 else "standard",
        "ef_unstable":        (5 <= n_actives < 20),
        "ef_not_reported":    (n_actives < 5),
        "ci_underpowered":    (20 <= n_actives < 50),
        "ci_not_reported":    (n_actives < 20),
    }


# ---------- 3. EF@10% and per-split metrics ----------
def compute_ef_at_10pct(y_true, y_score):
    n = len(y_true)
    k = max(1, int(np.ceil(0.10 * n)))
    top_k_idx = np.argsort(y_score)[-k:]
    hits = int(y_true[top_k_idx].sum())
    rate_in_top = hits / k
    rate_overall = float(y_true.sum()) / n
    ef = (rate_in_top / rate_overall) if rate_overall > 0 else None
    return ef, hits, k


def scaffold_stats_for(idxs, scaffold_map):
    scafs = [scaffold_map.get(str(i), "") for i in idxs]
    c = Counter(scafs)
    return len(c), (max(c.values()) if c else 0)


def split_metrics(model, X, y, idxs, scaffold_map):
    preds = model.predict_proba(X)[:, 1]
    n_actives = int(y.sum())
    n_inactives = int(len(y) - n_actives)
    top_n_idx = np.argsort(preds)[-n_actives:] if n_actives > 0 else []
    top_n_hits = int(y[top_n_idx].sum()) if n_actives > 0 else 0
    ef, ef_hits, ef_bin_n = compute_ef_at_10pct(y, preds)
    n_scaf, largest = scaffold_stats_for(idxs, scaffold_map)
    return {
        "pr_auc": float(average_precision_score(y, preds)),
        "roc_auc": float(roc_auc_score(y, preds)),
        "ef_at_10pct": ef,
        "ef_at_10pct_hits": ef_hits,
        "ef_at_10pct_bin_n": ef_bin_n,
        "top_n_recovery": (float(top_n_hits / n_actives) if n_actives > 0 else None),
        "test_n_actives": n_actives,
        "test_n_inactives": n_inactives,
        "n_scaffold_groups": n_scaf,
        "largest_scaffold_group_size": largest,
    }


cv_data = json.loads(Path("../splits/cv_scaffold_groups.json").read_text())
scaffold_map = cv_data["scaffold_of_compound"]

val_m  = split_metrics(selected_model, X_val,  y_val,  splits["indices"]["validation"], scaffold_map)
test_m = split_metrics(selected_model, X_test, y_test, splits["indices"]["test"],       scaffold_map)


# ---------- 4. Dummy baseline (computed, not hard-coded) ----------
dummy_pr_auc        = float(y_test.sum() / len(y_test))   # constant-prediction PR-AUC
dummy_roc_auc       = 0.5
dummy_top_n_recovery = float(y_test.sum() / len(y_test))  # random expectation


# ---------- 5. Assemble evaluation table ----------
sim_flags = small_n_flags(test_m["test_n_actives"])

def row(model, split, m, flags, notes):
    return {
        "model": model,
        "split": split,
        "pr_auc": m.get("pr_auc"),
        "roc_auc": m.get("roc_auc"),
        "ef_at_10pct": m.get("ef_at_10pct"),
        "top_n_recovery": m.get("top_n_recovery"),
        **flags,
        "test_n_actives": m.get("test_n_actives"),
        "test_n_inactives": m.get("test_n_inactives"),
        "n_scaffold_groups": m.get("n_scaffold_groups"),
        "largest_scaffold_group_size": m.get("largest_scaffold_group_size"),
        "notes": notes,
    }


# ===============================================================
# PHASE 3: Similarity baseline + Bootstrap CIs
# ===============================================================
from rdkit import DataStructs

# --- 1. Similarity-only baseline
train_act_mask = (curated.iloc[splits["indices"]["train"]]["activity_class"] == "active").values
train_idx_arr = np.array(splits["indices"]["train"])
train_active_idx = train_idx_arr[train_act_mask]

morgan_sim = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
train_active_fps = [
    morgan_sim.GetFingerprint(Chem.MolFromSmiles(curated.iloc[i]["canonical_smiles"]))
    for i in train_active_idx
]

sim_scores = []
for i in splits["indices"]["test"]:
    mol = Chem.MolFromSmiles(curated.iloc[i]["canonical_smiles"])
    fp = morgan_sim.GetFingerprint(mol)
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_active_fps)
    sim_scores.append(float(max(sims)) if sims else 0.0)

sim_scores = np.array(sim_scores)
sim_pr  = float(average_precision_score(y_test, sim_scores))
sim_roc = float(roc_auc_score(y_test, sim_scores))
n_act = int(y_test.sum())
sim_top_n = float(y_test[np.argsort(sim_scores)[-n_act:]].sum() / n_act) if n_act else None
sim_ef, _, _ = compute_ef_at_10pct(y_test, sim_scores)

sim_m = {
    "pr_auc": sim_pr, "roc_auc": sim_roc,
    "ef_at_10pct": sim_ef, "top_n_recovery": sim_top_n,
    "test_n_actives": int(y_test.sum()),
    "test_n_inactives": int(len(y_test) - y_test.sum()),
    "n_scaffold_groups": test_m["n_scaffold_groups"],
    "largest_scaffold_group_size": test_m["largest_scaffold_group_size"],
}
print(f"Similarity-only Baseline: PR-AUC {sim_pr:.3f}, ROC-AUC {sim_roc:.3f}")

# --- 2. Bootstrap CIs (1000 resamples) for the selected model
rng = np.random.default_rng(42)
test_preds = selected_model.predict_proba(X_test)[:, 1]
pr_boot, roc_boot = [], []

for _ in range(1000):
    idx = rng.integers(0, len(y_test), len(y_test))
    if y_test[idx].sum() == 0 or y_test[idx].sum() == len(idx):
        continue  # skip degenerate resample
    pr_boot.append(average_precision_score(y_test[idx], test_preds[idx]))
    roc_boot.append(roc_auc_score(y_test[idx], test_preds[idx]))

if len(pr_boot) >= 100:
    test_m["pr_auc_ci_low"]  = float(np.percentile(pr_boot, 2.5))
    test_m["pr_auc_ci_high"] = float(np.percentile(pr_boot, 97.5))
    test_m["roc_auc_ci_low"] = float(np.percentile(roc_boot, 2.5))
    test_m["roc_auc_ci_high"] = float(np.percentile(roc_boot, 97.5))
    test_m["bootstrap_n_valid"] = len(pr_boot)
    print(f"Test PR-AUC 95% CI:  [{test_m['pr_auc_ci_low']:.3f}, {test_m['pr_auc_ci_high']:.3f}]")
    print(f"Test ROC-AUC 95% CI: [{test_m['roc_auc_ci_low']:.3f}, {test_m['roc_auc_ci_high']:.3f}]")
else:
    test_m["pr_auc_ci_low"] = test_m["pr_auc_ci_high"] = None
    test_m["roc_auc_ci_low"] = test_m["roc_auc_ci_high"] = None
    test_m["bootstrap_n_valid"] = len(pr_boot)


eval_rows = [
    row(
        "Dummy", "test",
        {"pr_auc": dummy_pr_auc, "roc_auc": dummy_roc_auc,
         "ef_at_10pct": None, "top_n_recovery": dummy_top_n_recovery,
         "test_n_actives": test_m["test_n_actives"],
         "test_n_inactives": test_m["test_n_inactives"],
         "n_scaffold_groups": test_m["n_scaffold_groups"],
         "largest_scaffold_group_size": test_m["largest_scaffold_group_size"]},
        small_n_flags(test_m["test_n_actives"]),
        "Constant-prediction baseline; PR-AUC equals positive-class fraction.",
    ),
    row("Similarity-only", "test", sim_m, sim_flags, "Max Tanimoto to training actives; no model."),
    row(
        "Docking-only", "test",
        {"test_n_actives": test_m["test_n_actives"],
         "test_n_inactives": test_m["test_n_inactives"],
         "n_scaffold_groups": test_m["n_scaffold_groups"],
         "largest_scaffold_group_size": test_m["largest_scaffold_group_size"]},
        sim_flags,
        "PENDING docking implementation — populate from scripts/dock_ligands.py.",
    ),
    row(
        "Selected RF", "validation",
        val_m,
        small_n_flags(val_m["test_n_actives"]),
        "Selection split; defines the ML_rank_score reference distribution.",
    ),
    row(
        "Selected RF", "test",
        test_m,
        small_n_flags(test_m["test_n_actives"]),
        "Headline test result.",
    ),
]

col_order = [
    "model", "split",
    "pr_auc", "roc_auc", "ef_at_10pct", "top_n_recovery",
    "pr_auc_ci_low", "pr_auc_ci_high", "roc_auc_ci_low", "roc_auc_ci_high",
    "top_recovery_mode", "ef_unstable", "ef_not_reported",
    "ci_underpowered", "ci_not_reported",
    "test_n_actives", "test_n_inactives",
    "n_scaffold_groups", "largest_scaffold_group_size",
    "bootstrap_n_valid", "notes",
]

df_eval = pd.DataFrame(eval_rows).reindex(columns=col_order)
df_eval.to_csv("../results/evaluation_table.csv", index=False)

print(f"\nEvaluation table saved: {len(df_eval)} rows × {len(col_order)} columns")
print(df_eval[["model", "split", "pr_auc", "top_recovery_mode",
               "ef_unstable", "ef_not_reported", "ci_not_reported"]].to_string(index=False))

Model saved to ../models/selected_model_rf.pkl
Model config saved to ../models/model_config.json
Similarity-only Baseline: PR-AUC 0.735, ROC-AUC 0.709
Test PR-AUC 95% CI:  [0.435, 1.000]
Test ROC-AUC 95% CI: [0.456, 1.000]

Evaluation table saved: 5 rows × 21 columns
          model      split   pr_auc top_recovery_mode  ef_unstable  ef_not_reported  ci_not_reported
          Dummy       test 0.333333             top_N         True            False             True
Similarity-only       test 0.734844             top_N         True            False             True
   Docking-only       test      NaN             top_N         True            False             True
    Selected RF validation 0.980909             top_N         True            False             True
    Selected RF       test 0.780612             top_N         True            False             True


In [5]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
import numpy as np
import pandas as pd

# 1. Re-initialize the generator (same as Cell 1)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# 2. Get RDKit fingerprint objects for the training set
train_smiles = curated.iloc[splits["indices"]["train"]]["canonical_smiles"]
train_fps = [morgan_gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in train_smiles]

# 3. Calculate LOO training distances
train_distances = []
for i, fp in enumerate(train_fps):
    # Calculate Tanimoto similarity to all other training compounds
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    sims.pop(i) # Remove self-similarity (which is 1.0)
    sims.sort(reverse=True)
    
    # Mean Jaccard distance (1 - Tanimoto) to the 5 nearest neighbors
    dist = np.mean([1 - s for s in sims[:5]])
    train_distances.append(dist)

# Locked threshold: 95th percentile of the training LOO distances
ad_threshold = np.percentile(train_distances, 95)
print(f"AD Threshold (95th percentile): {ad_threshold:.4f}\n")

# 4. Calculate AD for the 28 Rosemary compounds
rosemary = pd.read_csv("../data/cultivar_compound_catalogue.csv").drop_duplicates("compound_id")
ad_results = []

for _, row in rosemary.iterrows():
    mol = Chem.MolFromSmiles(row["canonical_smiles"])
    if mol is None:
        continue
    fp = morgan_gen.GetFingerprint(mol)
    
    # Calculate similarity to training set
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    sims.sort(reverse=True)
    
    # Mean Jaccard distance to 5 nearest training neighbors
    dist = np.mean([1 - s for s in sims[:5]])
    
    flag = "within_AD" if dist <= ad_threshold else "extrapolation"
    ad_results.append({
        "compound_id": row["compound_id"], 
        "compound_name": row["compound_name"],
        "ad_distance": dist, 
        "ad_flag": flag
    })

df_ad = pd.DataFrame(ad_results)

# 5. Report results
print("AD Summary for Rosemary Catalogue:")
print(df_ad["ad_flag"].value_counts().to_string())
print("\nCompounds flagged as Extrapolation (Distance > Threshold):")
print(df_ad[df_ad["ad_flag"] == "extrapolation"][["compound_name", "ad_distance"]].sort_values("ad_distance", ascending=False).head(10).to_string(index=False))

# Save the results for Day 3 scoring
df_ad.to_csv("../results/ad_results.csv", index=False)
print("\nSaved AD results to ../results/ad_results.csv")

AD Threshold (95th percentile): 0.7330

AD Summary for Rosemary Catalogue:
ad_flag
extrapolation    28

Compounds flagged as Extrapolation (Distance > Threshold):
      compound_name  ad_distance
        1,8-Cineole     0.926550
           Camphene     0.908352
            Myrcene     0.903000
        beta-Pinene     0.897331
Caryophyllene oxide     0.895391
     gamma-Eudesmol     0.893818
            Camphor     0.892894
           Linalool     0.889891
            Borneol     0.888856
    gamma-Terpinene     0.887453

Saved AD results to ../results/ad_results.csv
